In [3]:
from transfer_train_pipeline import run_transfer_training
import yaml

with open("configs/maskrcnn_resnet50_fpn_synth.yaml", "r") as f:
    cfg = yaml.safe_load(f)

#model, best_val, best_epoch, out_dir = run_transfer_training(cfg)
#(out_dir, best_val, best_epoch)


In [4]:
import os
import torch
import datetime as dt

from models import MODEL_REGISTRY
from test_eval import build_test_dataloader, run_test_evaluation
from plotting_module import run_all_plots_for_experiment

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EVAL_SUITES = {
    "real_test": "../../../datasets/RealBubbles.coco-segmentation/test",
    "synth_test": "../../../BubANN_GAN.v1i.coco-segmentation/test"
}

eval_suite = "real_test"

# Force all comparisons to use the same test set:
#cfg["dataset"]["test"] = EVAL_SUITES[eval_suite]

# 2) Build test loader (uses cfg["dataset"]["test"])
test_dl = build_test_dataloader(cfg)

model_to_analyse = "maskrcnn_resnet50_fpn_transfer_real_20251218_1643"

# 3) Load model + weights
model = MODEL_REGISTRY[cfg["model"]["type"]](cfg["model"]["num_classes"])
weights_path = os.path.join("outputs", f"{model_to_analyse}", "model_best.pth")
model.load_state_dict(torch.load(weights_path, map_location=device))
model.to(device)

# 4) Evaluate -> writes results.json + csv + examples
exp_name = cfg["experiment"]["name"]
out_dir = os.path.join("results", eval_suite, model_to_analyse)

results = run_test_evaluation(model, test_dl, device, cfg, out_dir, visualize=False)

# 5) Generate plots
run_all_plots_for_experiment(out_dir)

out_dir, results


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!

=== Test Metrics Summary ===
precision: 0.9626737594628003
recall: 0.9773273311376545
f1: 0.9695226748027066
dice: 0.9695226747805092
iou: 0.9409958345199522
avg_bubbles: 10.366666666666667
mean_area: 4730.225763770084
median_area: 4542.123333333333
gt_avg_bubbles: 10.473333333333333
gt_mean_area: 4586.639139724844
gt_median_area: 4434.616666666667
count_error_mean: -0.10666666666666667
abs_count_error_mean: 0.21333333333333335
num_images: 150
num_empty_gt: 0
num_empty_pred: 0
num_empty_both: 0
avg_infer_time: 0.11482743899027506
fps: 8.70871987386823

Visual examples saved in: results/real_test/maskrcnn_resnet50_fpn_transfer_real_20251218_1643/examples
JSON summary saved in: results/real_test/maskrcnn_resnet50_fpn_transfer_real_20251218_1643/results.json
CSV saved in: results/real_test/maskrcnn_resnet50_fpn_transfer_real_20251218_1643/test_semantic_metrics.csv
Generating plots for maskrcnn_resnet50_fpn

('results/real_test/maskrcnn_resnet50_fpn_transfer_real_20251218_1643',
 EvalResults(semantic_metrics={'precision': np.float64(0.9626737594628003), 'recall': np.float64(0.9773273311376545), 'f1': np.float64(0.9695226748027066), 'dice': np.float64(0.9695226747805092), 'iou': np.float64(0.9409958345199522)}, instance_metrics={'avg_bubbles': np.float64(10.366666666666667), 'mean_area': np.float64(4730.225763770084), 'median_area': np.float64(4542.123333333333), 'gt_avg_bubbles': np.float64(10.473333333333333), 'gt_mean_area': np.float64(4586.639139724844), 'gt_median_area': np.float64(4434.616666666667), 'count_error_mean': np.float64(-0.10666666666666667), 'abs_count_error_mean': np.float64(0.21333333333333335), 'num_images': 150, 'num_empty_gt': 0, 'num_empty_pred': 0, 'num_empty_both': 0}, timing={'avg_infer_time': np.float64(0.11482743899027506), 'fps': 8.70871987386823}, config_used={'score_thresh': 0.5, 'min_area': 50, 'mask_thresh': 0.5}, examples_dir='results/real_test/maskrcnn_re